# PDL Practical Lab 

Each section has:
- **Concept:** What you're learning
- **We Do:** Worked example to run
- **You Do:** Your turn to write the code!

**To maximise learning:** Try not to copy-paste, and type the code yourself as much as possible.

---

## Setup & Configuration (Run this cell with Shift + Enter)

In [ ]:
import os
os.environ['OLLAMA_HOST'] = 'http://ollama:11434'

import litellm
litellm.api_base = 'http://ollama:11434'

import requests
try:
    response = requests.get('http://ollama:11434/api/tags')
    models = response.json()['models']
    print("✓ Connected to Ollama")
    print(f"✓ Available models: {len(models)}")
    for model in models:
        print(f"   - {model['name']}")
except Exception as e:
    print(f"Status: Waiting for Ollama to start...")

%load_ext pdl.pdl_notebook_ext

---
# Part 0: How PDL Works

## Concept

PDL automatically manages context for you. Every block you write becomes part of an implicit conversation:

1. **You arrange blocks** in order (text, model calls, etc.)
2. **PDL combines them** into one conversation
3. **Models see everything** that came before automatically


### Simple Example

```yaml
text:
- "You are a helpful tutor.\n"
- "Student asks: What is a list?\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 150
```

The model sees both text lines as part of one conversation. No manual context management needed.

## We Do: Worked Example

Run the cell below to see context accumulation in action.

In [ ]:
%%pdl --reset-context
description: Simple model call showing context accumulation

text:
- "You are a helpful Python tutor.\n"
- "A student asks: What is a list comprehension?\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 150

## You Do: Your First Program

**Your Challenge:** Write a PDL program that:
1. Defines a role ("You are a [something] expert")
2. Asks a question related to that role
3. Gets a response from the model

**Success Criteria:**
- The cell runs without errors
- The model responds based on your role and question

**Template:**
```yaml
%%pdl --reset-context
description: [Replace with: "My first PDL program"]

text:
- "You are a [PICK A ROLE].\n"
- "Question: [PICK A QUESTION].\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 100
```

**Example Ideas:**
- "You are a physics expert" + "What is gravity?"
- "You are a chef" + "How do I make pasta?"
- "You are a historian" + "When did Rome fall?"

Write your code in the cell below:

In [ ]:
# YOUR CODE HERE
# Delete these comments and write your PDL program

---
# Part 1: Using the Model (with parameters)

## Concept

Control how the model behaves:

| Parameter | What It Does | Range | Example |
|-----------|-------------|-------|----------|
| **temperature** | How creative/random | 0 to 1 | 0.1 = focused, 0.9 = creative |
| **max_tokens** | Max response length | 1 to 4000 | 100 = short, 500 = long |

```yaml
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.3      # Low = focused
    max_tokens: 200       # How long?
```

## We Do: Worked Example

Compare the same prompt with different parameters.

In [ ]:
%%pdl --reset-context
description: Controlled output with parameters

text:
- "Generate a creative function name for sorting.\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.3
    max_tokens: 50

## You Do: Experiment with Parameters

**Your Challenge:** Write a program that asks the model the same question twice:
1. **First call:** Use `temperature: 0.1` (very focused)
2. **Second call:** Use `temperature: 0.9` (very creative)

Ask for "a silly name for a function." Watch how temperature changes the creativity.

**Success Criteria:**
- Both model calls run
- The creative response (temp 0.9) is different from the focused one (temp 0.1)

**Hint:** Try keeping both model calls in the same cell. PDL will accumulate both responses.

Write your code below:

In [ ]:
# YOUR CODE FOR PART 1 CHALLENGE

---
# Part 2: Reusable Values with `defs:`

## Concept

**`defs:`** lets you define values once, use them many times.

```yaml
defs:
  language: Python
  topic: "What is a list?"

text:
- "Language: ${language}\n"
- "Question: ${topic}\n"
```

When PDL sees `${language}`, it replaces it with `"Python"`.

**Why it matters:**
- Define setup once at the top
- Edit one place to change the entire program
- Easy to experiment with different values

## We Do: Worked Example

See how changing one variable impacts the entire output.

In [ ]:
%%pdl --reset-context
description: Using defs to define reusable values

defs:
  language: French
  topic: "What is recursion?"

text:
- "Explain in ${language}: ${topic}\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.6
    max_tokens: 150

## You Do: Combine Two Variables

**Your Challenge:** Write a PDL program that:
1. Defines 2 variables in `defs:` (pick any values you like)
2. Uses BOTH variables in questions to the model
3. The model generates an answer combining both values

**Success Criteria:**
- Cell runs without errors
- Both variables appear in the output
- Model's response shows it understood both variables

**Example topic combinations:**
- `cuisine: Italian` + `season: summer` -> "Design an Italian summer menu"
- `language: Python` + `time_limit: 5 minutes` -> "Write Python code to do X in 5 minutes"
- `style: funny` + `topic: socks` -> "Write a funny article about socks"


Write your code below:

In [ ]:
# YOUR CODE FOR PART 2 CHALLENGE

---
# Part 3: Structured Output with `parser: json` & `spec:`

## Concept

**Problem:** Models sometimes add extra text around JSON:
```
Here's the JSON:
{"name": "Alice", "age": 30}
Hope this helps!
```

**Solution:** Add `parser: json` to extract it, and `spec:` to validate structure:

```yaml
- model: ollama/granite-code:3b
  parser: json
  spec:                    # Define expected fields
    name: string
    age: integer
  def: result              # Store as variable
```

PDL automatically:
1. Extracts JSON from the response
2. Validates it has the correct fields and types
3. Saves it as `result` for later use

## We Do: Worked Example

Ask the model for structured data (JSON), validate it, then use the fields.

In [ ]:
%%pdl --reset-context
description: JSON validation and structured output

text:
- lastOf:
  - "Generate me a Python function specification as JSON.\n"
  - "EXAMPLE format: {\"name\": \"is_prime\", \"return_type\": \"bool\", \"difficulty\": \"medium\"}\n"
  - "Create a DIFFERENT function spec. Return ONLY valid JSON.\n"
  - model: ollama/granite-code:3b
    parameters:
      temperature: 0.5
      max_tokens: 120
    parser: json
    spec:
      name: string
      return_type: string
      difficulty: string
    def: function_spec
- "\n=== FUNCTION SPEC ===\n"
- "Function: ${function_spec.name}\n"
- "Returns: ${function_spec.return_type}\n"
- "Difficulty: ${function_spec.difficulty}\n"

## You Do: Generate Your Own Structured Data

**Your Challenge:** Write a PDL program that:
1. Asks the model to generate JSON with 3 fields (pick any topic)
2. Validates the structure with `spec:`
3. Extracts and displays the fields neatly

**Success Criteria:**
- Cell runs without errors
- JSON is successfully parsed (no "undefined" errors)
- All 3 fields appear in the formatted output

**Create a topic for your JSON:**

**Example A: Restaurant Review**
- Fields: name (string), rating (integer 1-5), cuisine (string)

**Example B: Book Summary**
- Fields: title (string), author (string), pages (integer)

**Example C: Movie Record**
- Fields: title (string), year (integer), score (integer 1-10)

Write your code below:

In [ ]:
# YOUR CODE FOR PART 3 CHALLENGE

**Experiment:** Try changing the PDL prompt to be less explicit about JSON format. What happens?

---
# Part 4: Hiding Intermediate Steps with `contribute:`

## Concept

By default, everything appears in the output. Sometimes you want intermediate steps hidden from the user.

Use `contribute:` to control visibility:

| `contribute:` | Visible to User? | Visible in Context? | Use For |
|-------------|-----------------|-------------------|----------|
| (default) | Yes | Yes | Final output |
| `[context]` | **No** | Yes | Setup, hidden work |
| `[]` | **No** | **No** | Scratch work |

**Example Pipeline:**
```yaml
# Step 1: Hidden from user, visible to next step
- model: code_generator
  contribute: [context]
  def: generated_code

# Step 2: Reviews the hidden code, visible to user
- model: code_reviewer
  # default behavior = visible
```

## We Do: Worked Example

Two-step pipeline: generate code (hidden), evaluate it (visible).

In [ ]:
%%pdl --reset-context
description: Two-step pipeline with output control

# STEP 1: Generate a name (hidden from user)
text:
- "Generate a creative function name for sorting items.\n"
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 50
  contribute: [context]
  def: generated_name

# STEP 2: Evaluate the name and show result
text:
- lastOf:
  - "Rate this function name. Is it good or bad? Why?\n"
  - model: ollama/granite-code:3b
    parameters:
      temperature: 0.3
      max_tokens: 100
    def: evaluation
- "\n=== EVALUATION ===\n"
- "${evaluation}\n"

## You Do: Build a Two-Step Pipeline

**Your Challenge:** Create a program that:
1. **Step 1:** Generate something (hidden with `contribute: [context]`)
2. **Step 2:** Process/evaluate it (visible to user)

**Success Criteria:**
- Cell runs without errors
- User sees ONLY step 2's output
- Step 2 references step 1 (proving the hidden step worked)

**Pick a pipeline topic:**

**Option A: Code Generation → Quality Check**
1. Generate a Python function (hidden)
2. Check if it's efficient (visible)

**Option B: Story Generation → Critique**
1. Generate a short story opening (hidden)
2. Critique the writing style (visible)

**Option C: Idea Generation → Evaluation**
1. Generate a startup idea (hidden)
2. Evaluate its feasibility (visible)

**Template:**
```yaml
%%pdl --reset-context
description: Two-step hidden+visible pipeline

# STEP 1: Hidden
text:
- "[GENERATE SOMETHING]\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 100
  contribute: [context]
  def: generated

# STEP 2: Visible
text:
- "[EVALUATE THE ABOVE]\n"

- model: ollama/granite-code:3b
  parameters:
    temperature: 0.3
    max_tokens: 100
  def: result

# STEP 3: Show result
text:
- "\n=== RESULT ===\n"
- "${result}\n"
```

Write your code below:

In [ ]:
# YOUR CODE FOR PART 4 CHALLENGE

**Experiment:** Try removing `contribute: [context]` from step 1. Now the user sees BOTH steps. Which is cleaner? Why would you hide intermediate steps?

---
# Part 5: Build Your First Real System

## Challenge: Multi-Step Code Review Pipeline

Combine everything you learned into a working system.

## We Do: Worked Example

Three-step pipeline: generate code (hidden) → review it (visible).

In [ ]:
%%pdl --reset-context
description: Code generation and review pipeline

defs:
  problem: "Write a function to calculate the factorial of a number"

# STEP 1: Generate code (hidden from user)
text:
- "Write Python code for: ${problem}\n"
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.3
    max_tokens: 200
  contribute: [context]
  def: generated_code

# STEP 2: Review the code and show results
text:
- lastOf:
  - "Review this code. Is it correct? Is it well-written? Suggest improvements.\n"
  - model: ollama/granite-code:3b
    parameters:
      temperature: 0.3
      max_tokens: 150
    def: review
- "\n=== CODE REVIEW ===\n"
- "Problem: ${problem}\n"
- "Review:\n"
- "${review}\n"

## You Do: Your Real System

**Your Challenge:** Build a pipeline for something meaningful to you, or just use the examle.

**Must include:**
1. At least one `defs:` variable for setup
2. At least one hidden step with `contribute: [context]`
3. At least one visible processing step
4. A final formatted output

**Success Criteria:**
- Cell runs without errors
- Each step uses results from previous steps
- Output is clean and useful
- You can edit one `defs:` value and the whole system adapts

**Example Pipelines:**

**Example A: API Documentation Generator** (3-step)
1. Hidden: Generate API endpoint spec
2. Hidden: Generate example request/response
3. Visible: Output formatted documentation

**Example B: Essay Grading System** (4-step)
1. Hidden: Analyze essay quality
2. Hidden: Generate suggestions
3. Visible: Check if suggestions improve quality
4. Visible: Show grade + feedback

**Example C: Code Optimization Pipeline** (3-step)
1. Hidden: Generate initial code
2. Hidden: Optimize for performance
3. Visible: Compare the two versions

**Example D: Your Own Idea**
- Think of something useful in your field
- Map it to 3+ steps
- Build it below

Write your code below:

In [ ]:
# YOUR CAPSTONE PROJECT CODE

---
# Summary: What You Learned

## Core Concepts

| Concept | Purpose | Example |
|---------|---------|----------|
| **Implicit Context** | PDL automatically threads conversation | Just write blocks in order |
| **Parameters** | Control model behavior | `temperature: 0.3` for focused output |
| **defs:** | Reusable values | Define once, use anywhere with `${variable}` |
| **def:** | Store results | Save model output for later use |
| **parser: json** | Extract structured data | Automatically parse JSON from messy output |
| **spec:** | Validate structure | Ensure JSON has required fields |
| **contribute:** | Control visibility | Hide intermediate steps with `[context]` |

## Next Steps

### Experiment More
- Modify `defs:` values in any example
- Change `temperature` and `max_tokens`
- Swap models (try `granite-code:3b` for faster)
- Add more steps to pipelines

### Go Deeper
- Learn `if/then/else` for conditional logic
- Learn `repeat:` for loops
- Learn `for:` for iterating over lists

### Advanced Topics
- Using multiple models in one pipeline
- Chaining validations with multiple `parser:` types

## Resources

- **Official Tutorial:** [PDL GitHub](https://github.com/IBM/prompt-declaration-language/tree/main/docs)
- **All Examples:** Check the [official examples folder](https://github.com/IBM/prompt-declaration-language/tree/main/examples)